# Nile-ViT — M1: Prithvi-EO-2.0 Smoke Test

This notebook is the **architectural de-risk** for the whole Nile-ViT project. It proves three things:

1. **TerraTorch installs cleanly** on a fresh Colab T4
2. **Prithvi-EO-2.0-300M weights download** from Hugging Face and load into memory
3. **A forward pass + LoRA attachment work** — confirming the `frozen Prithvi + LoRA adapters` recipe in the PRD is viable

If everything in this notebook passes, **M1 is closed** and Phase 2 (data pipeline) is unblocked.

---

### Before you run

- **Runtime → Change runtime type → T4 GPU** (this is essential — Prithvi-300M won't fit in CPU memory comfortably)
- Have your **Hugging Face token** ready (the `hf_...` one from `https://huggingface.co/settings/tokens`)
- The install step takes 5–10 minutes; the whole notebook ~15 minutes end-to-end


## 1. Environment check

Verify we're on a GPU runtime before doing anything expensive.


In [ ]:
import torch
import platform

print(f"Python:       {platform.python_version()}")
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA built:   {torch.version.cuda}")
print(f"CUDA avail:   {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. In Colab: Runtime → Change runtime type → T4 GPU, "
        "then run all cells again."
    )

print(f"GPU:          {torch.cuda.get_device_name(0)}")
print(f"VRAM total:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"VRAM free:    {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB")


## 2. Install dependencies

`terratorch` is the IBM/NASA library that gives us pretrained Prithvi backbones with a clean API. It pulls in a lot transitively (timm, lightning, segmentation-models-pytorch, etc.), so this cell takes 5–10 minutes the first time.

> **Tip**: if Colab asks to restart the runtime after install — **say yes**, then re-run from the GPU check cell onward.


In [ ]:
%pip install -q --upgrade pip
%pip install -q --force-reinstall --no-deps "numpy>=2.0,<2.1"
%pip install -q "terratorch>=1.0.1,<2.0" "peft>=0.13" "huggingface_hub>=0.25"
%pip uninstall -y torchao
print("✅ Install complete — RESTART RUNTIME before running cell 5")

In [ ]:
# Verify versions
import importlib.metadata as md_

for pkg in ["torch", "terratorch", "peft", "transformers", "timm", "huggingface_hub"]:
    try:
        ver = md_.version(pkg)
        print(f"  {pkg:20s} {ver}")
    except md_.PackageNotFoundError:
        print(f"  {pkg:20s} NOT INSTALLED")


## 3. Hugging Face authentication

Prithvi-EO-2.0 weights live on Hugging Face Hub. The repo is public but we still need to be logged in for the API to download.

**Recommended:** add `HF_TOKEN` to **Colab Secrets** (the key icon 🔑 in the left sidebar) once, then this notebook will pick it up automatically every time. Otherwise paste your token when prompted.


In [ ]:
import os
from huggingface_hub import login

hf_token = None

# Try Colab Secrets first
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    print("Using HF_TOKEN from Colab Secrets")
except Exception:
    pass

# Fallback to env var
if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")
    if hf_token:
        print("Using HF_TOKEN from environment")

# Final fallback: prompt
if not hf_token:
    from getpass import getpass
    hf_token = getpass("Paste your Hugging Face token (starts with hf_): ").strip()

login(token=hf_token, add_to_git_credential=False)
print("✅ Hugging Face login OK")


## 4. Load Prithvi-EO-2.0-300M

We load the 300M-parameter variant via TerraTorch's `BACKBONE_REGISTRY`. First call downloads ~1.2 GB of weights from Hugging Face into Colab's cache (`~/.cache/huggingface/`).

The backbone is pretrained on Harmonized Landsat-Sentinel-2 (HLS) imagery: 6 bands (Blue, Green, Red, NIR-narrow, SWIR-1, SWIR-2), trained on temporal sequences of 3 frames at 30 m resolution.


In [ ]:
import terratorch
from terratorch import BACKBONE_REGISTRY

print(f"TerraTorch version: {terratorch.__version__ if hasattr(terratorch, '__version__') else 'unknown'}")

# What backbones are registered?
prithvi_backbones = [name for name in BACKBONE_REGISTRY if "prithvi" in name.lower()]
print(f"\nAvailable Prithvi backbones ({len(prithvi_backbones)}):")
for b in prithvi_backbones[:20]:
    print(f"  - {b}")


In [ ]:
# Build Prithvi-EO-2.0-300M with pretrained weights
backbone = BACKBONE_REGISTRY.build(
    "prithvi_eo_v2_300",
    pretrained=True,
    bands=["BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2"],
    num_frames=3,
)

backbone = backbone.cuda().eval()
print(f"✅ Loaded {type(backbone).__name__}")
print(f"   Total parameters: {sum(p.numel() for p in backbone.parameters()):,}")
print(f"   On device:        {next(backbone.parameters()).device}")


## 5. Forward pass with dummy input

A single forward pass with a randomly-initialized tensor in the shape Prithvi expects: `(batch, channels=6, time=3, height=224, width=224)`.

If this works, the backbone is correctly loaded and the device pipeline is functional.


In [ ]:
import torch

# Prithvi-EO-2.0 expected input shape
B, C, T, H, W = 1, 6, 3, 224, 224
dummy = torch.randn(B, C, T, H, W, device="cuda")

with torch.inference_mode():
    out = backbone(dummy)

# Output could be a tensor, tuple, or list of feature maps depending on backbone
if isinstance(out, torch.Tensor):
    print(f"Output (single tensor): shape={tuple(out.shape)}, dtype={out.dtype}")
elif isinstance(out, (list, tuple)):
    print(f"Output (sequence of {len(out)} tensors):")
    for i, t in enumerate(out):
        print(f"  [{i}] shape={tuple(t.shape)}, dtype={t.dtype}")
else:
    print(f"Output type: {type(out)}")
    print(out)

print(f"\n✅ Forward pass succeeded")
print(f"   VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB / "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 6. Attach LoRA adapters (the PEFT recipe)

This is the moment of truth for the architectural plan: wrap the frozen backbone with LoRA adapters via `peft`. If this works with target modules `qkv`, `mlp.fc1`, `mlp.fc2` at rank 16, the PRD's parameter-efficient fine-tuning strategy is viable.

We expect:
- **Total** params: ~300M (unchanged)
- **Trainable** params: ~3M (≈ 1% of total)


In [ ]:
# First, discover what attention/MLP modules exist in Prithvi
qkv_modules = []
mlp_modules = []
for name, module in backbone.named_modules():
    if name.endswith("qkv") or "qkv" in name.split(".")[-1]:
        qkv_modules.append(name)
    elif name.endswith(("fc1", "fc2", "mlp.fc1", "mlp.fc2")):
        mlp_modules.append(name)

print(f"Found {len(qkv_modules)} qkv modules (showing first 3):")
for n in qkv_modules[:3]:
    print(f"  {n}")

print(f"\nFound {len(mlp_modules)} mlp.fc modules (showing first 3):")
for n in mlp_modules[:3]:
    print(f"  {n}")

if not qkv_modules:
    print("\n⚠️  No 'qkv' modules found. Module names may differ in this Prithvi version.")
    print("   Inspect a few module names to find the right pattern:")
    for name, _ in list(backbone.named_modules())[:30]:
        print(f"  {name}")


In [ ]:
from peft import LoraConfig, get_peft_model

# Match the PRD's recipe: r=16, alpha=16, target qkv + MLP linears
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["qkv", "mlp.fc1", "mlp.fc2"],
)

lora_backbone = get_peft_model(backbone, lora_config)
lora_backbone.print_trainable_parameters()


In [ ]:
# Verify the LoRA-wrapped model still does a valid forward pass
with torch.inference_mode():
    out_lora = lora_backbone(dummy)

if isinstance(out_lora, torch.Tensor):
    print(f"LoRA output shape: {tuple(out_lora.shape)}")
elif isinstance(out_lora, (list, tuple)):
    print(f"LoRA output: sequence of {len(out_lora)} tensors")
    for i, t in enumerate(out_lora):
        print(f"  [{i}] shape={tuple(t.shape)}")
else:
    print(f"LoRA output type: {type(out_lora)}")

print(f"\n✅ LoRA forward pass succeeded")
print(f"   VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB / "
      f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 7. Summary — M1 acceptance

If you got here without errors, **M1 is closed**:

| Check | Status |
|---|---|
| TerraTorch installs on Colab T4 | ✅ |
| Prithvi-EO-2.0-300M weights download | ✅ |
| Forward pass on dummy input | ✅ |
| LoRA adapters attach with ~1% trainable params | ✅ |

This means the entire `Frozen Prithvi-EO-2.0 + LoRA adapters → fusion head` architecture from the PRD is technically viable on free Colab compute. Every downstream phase builds on what was just verified here.

### What you've validated about the PRD

- **§5.1** — visual encoder loads and runs ✅
- **§5.2** — LoRA config (r=16, α=16, target qkv + mlp.fc1/fc2) attaches ✅
- **§5.4** — parameter budget (~9.7M trainable target, ~3M from LoRA alone) tracks ✅
- **§10.1** — Stage A (Prithvi forward passes) fits on T4 ✅

### Next: Phase 2 begins

1. Commit this notebook to the repo at `notebooks/00_prithvi_smoke_test.ipynb`
2. Update CHANGELOG under "Unreleased" — note M1 closed
3. Begin **Phase 2 — Data Pipeline**: the HLS / ERA5 / CHIRPS / MODIS download scripts

The next ML work happens in the data pipeline, not the model. As the PRD's decision thresholds make clear: *the dataset is the most important artifact this project will produce*. Even if the model never beats the CNN baseline, a publishable Nile-region compound-event benchmark is publishable on its own.

---

> **One thing to know**: `transformers 5.x` was released in 2026 with breaking API changes. If you see an error from `peft` or terratorch about an attribute that "no longer exists on `PreTrainedModel`", we may need to pin `transformers<5` in `pyproject.toml`. We'll handle this if it comes up.
